# 02 — Data Cleaning and Feature Construction

1. Fix source-data typo (`Louisana` → `Louisiana`)
2. Add U.S. Census region for each state
3. Group 36 indicators into six policy domains and compute domain-average scores
4. Save `sdpi_processed.csv` for downstream notebooks

In [1]:
import pandas as pd
import numpy as np
from IPython.core.interactiveshell import InteractiveShell
InteractiveShell.ast_node_interactivity = "all"

## Load raw data

In [2]:
df = pd.read_csv("../data/SDPI_final_all_methods.csv")
df.shape

(51, 40)

## Step 1 — Fix the Louisiana typo

In [3]:
"Louisana" in df["State"].tolist()  # should be True

True

In [4]:
df["State"] = df["State"].replace({"Louisana": "Louisiana"})
"Louisiana" in df["State"].tolist()  # should now be True

True

## Step 2 — Map states to U.S. Census Bureau regions

In [5]:
REGION_MAP = {
    "Connecticut":"Northeast","Maine":"Northeast","Massachusetts":"Northeast",
    "New Hampshire":"Northeast","New Jersey":"Northeast","New York":"Northeast",
    "Pennsylvania":"Northeast","Rhode Island":"Northeast","Vermont":"Northeast",
    "Illinois":"Midwest","Indiana":"Midwest","Iowa":"Midwest","Kansas":"Midwest",
    "Michigan":"Midwest","Minnesota":"Midwest","Missouri":"Midwest","Nebraska":"Midwest",
    "North Dakota":"Midwest","Ohio":"Midwest","South Dakota":"Midwest","Wisconsin":"Midwest",
    "Alabama":"South","Arkansas":"South","Delaware":"South","District of Columbia":"South",
    "Florida":"South","Georgia":"South","Kentucky":"South","Louisiana":"South",
    "Maryland":"South","Mississippi":"South","North Carolina":"South","Oklahoma":"South",
    "South Carolina":"South","Tennessee":"South","Texas":"South","Virginia":"South",
    "West Virginia":"South",
    "Alaska":"West","Arizona":"West","California":"West","Colorado":"West","Hawaii":"West",
    "Idaho":"West","Montana":"West","Nevada":"West","New Mexico":"West","Oregon":"West",
    "Utah":"West","Washington":"West","Wyoming":"West",
}
df["region"] = df["State"].map(REGION_MAP)
assert df["region"].isna().sum() == 0, "Unmapped states!"
df["region"].value_counts()

region
South        17
West         13
Midwest      12
Northeast     9
Name: count, dtype: int64

## Step 3 — Define policy domains and compute domain scores

In [6]:
DOMAINS = {
    "Income & Cash Support": [
        "avg_monthly_ssi_payment","ssi_auto_enroll_medicaid","ssi_criteria_209",
        "initial_ar","reconsidered_ar","total_ar",
    ],
    "Medicaid & LTC": [
        "medicaid_eligibility_threshold","medically_needy","buy_in_working_people",
        "spousal_impoverishment","adl_medicaid_coverage_pct","private_ltci_per1000",
        "family_responsibility_class",
    ],
    "HCBS": [
        "hcbs_expenditure_ratio","hcbs_user_ratio","pct_ltss_hcbs_older_adults",
        "home_health_aides_per100","hcbs_presumptive_eligibility","smd_demonstration_projects",
    ],
    "Employment & VR": [
        "vr_spending_career","vr_spending_training","subminimum_wage_14c","ui_good_cause_caregiving",
    ],
    "ARP / Pandemic Relief": [
        "arp_caregiver_family_support","arp_waiting_list_diversion","arp_tech_telehealth",
        "arp_cross_sector_investments","arp_workforce_training","arp_quality_improvement",
    ],
    "Housing & Accessibility": [
        "livability_transportation","livability_housing","section_811_pra",
        "section_811_pct_disability","special_ed_policy_score","fema_shmp","caps",
    ],
}

for domain_name, cols in DOMAINS.items():
    df[f"domain_{domain_name}"] = df[cols].mean(axis=1)

domain_cols = [c for c in df.columns if c.startswith('domain_')]
print(f'Added {len(domain_cols)} domain columns')
df[['State'] + domain_cols].head()

Added 6 domain columns


,State,domain_Income & Cash Support,domain_Medicaid & LTC,domain_HCBS,domain_Employment & VR,domain_ARP / Pandemic Relief,domain_Housing & Accessibility
0,Alabama,0.459677,0.564308,0.184211,0.170357,0.057758,0.241377
1,Alaska,0.326042,0.487013,0.605263,0.536181,0.436168,0.358336
2,Arizona,0.608052,0.608195,0.578947,0.442597,0.571602,0.503309
3,Arkansas,0.425989,0.317376,0.403509,0.329694,0.401609,0.325266
4,California,0.655428,0.552351,0.605263,0.589179,0.661393,0.720599


## Step 4 — Save processed dataset

In [7]:
df.to_csv("../data/sdpi_processed.csv", index=False)
print(f"Saved: ../data/sdpi_processed.csv  ({df.shape[0]} rows × {df.shape[1]} cols)")

Saved: ../data/sdpi_processed.csv  (51 rows × 47 cols)
